# Workshop-Paper Entity Re-run Analysis

Entity-collapse diagrams for the workshop-paper entity re-run files, rendered in the **same naming and
style as `visualization.ipynb` / `agentic-rag-visualization.ipynb`**: overlaid line charts saved to
`workshop_entity_visualizations/<org>/<model>/<method>/{unique_entities,entity_similarity}_per_round.png`,
plus a grouped-bar **`collapse_by_simulation.png`** per multi-variant group.

Data folder is downloaded from Google Drive (`entity re-run for workshop paper-<timestamp>-3-001/`,
gitignored); cell 1 auto-discovers it by regex (and descends into the nested subfolder). The
`*.entities_by_round.jsonl` files are **GPT-5.2-tagged**; metrics recomputed to match `entity_extraction.py`.
The last cell asserts full coverage — every `.jsonl` in the dump must be plotted.

Two entity metrics per group (overlaid line charts) **and** the collapse-by-simulation bar chart
(matches `visualization.ipynb`: a question-round is "collapsed" when ALL runs share an identical
canonical entity set; bars = % collapsed at start / at end / mean % of rounds collapsed):

- **baseline** — 4 models (Qwen2.5-14B, Llama-3.1-8B, Mistral-7B, DeepSeek-R1-Distill-7B), Replace All/One/Search overlaid -> `<org>/<model>/baseline/`
- **comparison** — Qwen baseline (RA/RO/Search) + Agentic RAG -> `Qwen/Qwen2.5-14B-Instruct/comparison/`
- **rerun-paraphrase** — paraphrase RA/RO/Search for all 4 models -> `<org>/<model>/rerun-paraphrase/`
- **rerank** — Qwen full λ-sweep (0.1/0.5/0.7 oracle) -> `Qwen/.../rerank/`, plus Qwen λ=0.7 oracle-vs-desklib -> `Qwen/.../rerank_lambda0.7/`; non-Qwen models have only λ=0.7 so their `<org>/<model>/rerank/` is the oracle-vs-desklib focus
- **agentic_rag** — Qwen Agentic RAG standalone (line charts only; collapse-by-simulation needs ≥2 simulations) -> `Qwen/Qwen2.5-14B-Instruct/agentic_rag/`


In [1]:
import os, re, json
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binomtest

# -- Locate the entity re-run folder (gitignored; timestamp varies) by regex, descend one level --
_PAT = re.compile(r"entity re-run for workshop paper-.*")
_outer = sorted(d for d in os.listdir(".") if _PAT.fullmatch(d) and os.path.isdir(d))
if not _outer:
    raise FileNotFoundError("No 'entity re-run for workshop paper-*' folder in the repo root.")
_OUTER = _outer[-1]
_inner = [d for d in os.listdir(_OUTER) if os.path.isdir(os.path.join(_OUTER, d))]
BASE = os.path.join(_OUTER, _inner[0]) if _inner else _OUTER
print("Using entity re-run folder:", BASE)

USED = set()  # basenames of every jsonl actually plotted; the last cell checks full coverage

OUT_ROOT = "workshop_entity_visualizations"
# Colors match visualization.ipynb / agentic-rag-visualization.ipynb
COLORS = {"Replace All": "#1f77b4", "Replace One": "#ff7f0e", "Search": "#2ca02c", "Agentic RAG": "#9467bd"}
LAMBDA_COLORS = {"Rerank λ=0.1": "#1f77b4", "Rerank λ=0.5": "#ff7f0e",
                 "Rerank λ=0.7 (oracle)": "#2ca02c", "Rerank λ=0.7 (desklib)": "#d62728"}

# Some entity-rerun dumps name the DeepSeek baseline files with the served model name
# (deepseek_deepseek-r1-distill-qwen-7b) instead of the org/model token; tolerate either.
_DEEPSEEK_ALIASES = ("deepseek-ai_DeepSeek-R1-Distill-Qwen-7B", "deepseek_deepseek-r1-distill-qwen-7b")

def _resolve(fname):
    """Return the path to fname under BASE, falling back to the alternate DeepSeek token."""
    p = os.path.join(BASE, fname)
    if os.path.exists(p):
        return p
    a, b = _DEEPSEEK_ALIASES
    for alt in (fname.replace(a, b), fname.replace(b, a)):
        if alt != fname and os.path.exists(os.path.join(BASE, alt)):
            return os.path.join(BASE, alt)
    return p  # original (missing) path; caller reports MISSING

def wilson_pct(count, nobs):
    """(point%, lo%, hi%) — 95% Wilson score interval for a binomial proportion count/nobs."""
    if nobs == 0:
        return 0.0, 0.0, 0.0
    count, nobs = int(round(count)), int(round(nobs))
    ci = binomtest(count, nobs).proportion_ci(confidence_level=0.95, method="wilson")
    return 100.0 * count / nobs, 100.0 * ci.low, 100.0 * ci.high

def compute_entity_similarity(vectors):
    n = len(vectors)
    if n < 2:
        return 0.0
    sims = []
    for i in range(n):
        for j in range(i + 1, n):
            a, b = vectors[i], vectors[j]
            n1, n2 = np.linalg.norm(a), np.linalg.norm(b)
            sims.append(float(np.dot(a, b) / (n1 * n2)) if n1 > 0 and n2 > 0 else 0.0)
    return float(np.mean(sims))

def analyze_file(path):
    """Per-round mean unique-entity count and mean entity similarity (matches entity_extraction.py)."""
    pru, prs, mx = defaultdict(list), defaultdict(list), 0
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            rec = json.loads(line)
            rounds, counts, mapped = rec["round_numbers"], rec["entity_counts"], rec["mapped_entities"]
            mx = max(mx, len(rounds))
            for idx in range(len(rounds)):
                pru[idx].append(len(counts[idx]))
                runs = mapped[idx]
                vocab = sorted({e for run in runs for e in run})
                vi = {e: i for i, e in enumerate(vocab)}
                vecs = []
                for run in runs:
                    v = np.zeros(len(vocab))
                    for e in run:
                        v[vi[e]] = 1.0
                    vecs.append(v)
                prs[idx].append(compute_entity_similarity(vecs))
    return ([float(np.mean(pru[i])) for i in range(mx)], [float(np.mean(prs[i])) for i in range(mx)])

def compute_collapse(path):
    """Collapse-by-simulation metric (matches visualization.ipynb): a question-round is 'collapsed'
    when ALL runs in that round share an identical canonical entity set. Returns (point, lo, hi) %
    with **95% Wilson CIs** for three proportions: collapsed at the first round and at the last round
    (over questions), and collapsed question-rounds (pooled over all (question, round) cells)."""
    n = n_start = n_end = 0
    collapsed_rounds = total_rounds = 0
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            rec = json.loads(line)
            mapped = rec["mapped_entities"]
            nrounds = len(rec["round_numbers"])
            flags = [len({frozenset(run) for run in mapped[idx]}) == 1 for idx in range(nrounds)]
            if not flags:
                continue
            n += 1
            n_start += int(flags[0]); n_end += int(flags[-1])
            collapsed_rounds += sum(flags); total_rounds += len(flags)
    return {"start":  wilson_pct(n_start, n),
            "end":    wilson_pct(n_end, n),
            "rounds": wilson_pct(collapsed_rounds, total_rounds)}

def plot_group(series, out_subdir):
    """series: list of (label, color, filename). Saves the two overlaid entity charts
    (unique_entities/entity_similarity) in visualization.ipynb style under OUT_ROOT/out_subdir/."""
    data = {}
    for label, color, fname in series:
        path = _resolve(fname)
        if not os.path.exists(path):
            print("  MISSING:", fname); continue
        u, s = analyze_file(path)
        data[label] = (color, u, s)
        USED.add(os.path.basename(path))
    if not data:
        print(f"  no data for {out_subdir}; leaving existing PNGs untouched"); return
    os.makedirs(f"{OUT_ROOT}/{out_subdir}", exist_ok=True)
    for which, ylabel, title, fn in [
        ("u", "Unique Entities", "Unique Entities Per Round", "unique_entities_per_round.png"),
        ("s", "Entity Similarity", "Entity Similarity Per Round", "entity_similarity_per_round.png")]:
        fig, ax = plt.subplots(figsize=(10, 6))
        for label, (color, u, s) in data.items():
            vals = u if which == "u" else s
            ax.plot(range(1, len(vals) + 1), vals, linewidth=2.5, label=label, color=color)
        ax.set_xlabel("Round", fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(title, fontsize=14)
        ax.legend()
        plt.tight_layout()
        plt.savefig(f"{OUT_ROOT}/{out_subdir}/{fn}", dpi=150, bbox_inches="tight")
        plt.close(fig)
    print(f"wrote {out_subdir}: {list(data)}")

def plot_collapse_group(series, out_subdir):
    """series: list of (label, color, filename) — each file is one 'simulation'. Saves the
    grouped-bar 'Collapse by Simulation' chart (visualization.ipynb style) as
    OUT_ROOT/out_subdir/collapse_by_simulation.png. Error bars = 95% Wilson CI per proportion."""
    sims = []
    pts = {"start": [], "end": [], "rounds": []}
    los = {"start": [], "end": [], "rounds": []}
    his = {"start": [], "end": [], "rounds": []}
    for label, color, fname in series:
        path = _resolve(fname)
        if not os.path.exists(path):
            print("  MISSING:", fname); continue
        m = compute_collapse(path)
        sims.append(label)
        for k in ("start", "end", "rounds"):
            p, lo, hi = m[k]
            pts[k].append(p); los[k].append(lo); his[k].append(hi)
        USED.add(os.path.basename(path))
    if not sims:
        print(f"  no data for {out_subdir}/collapse_by_simulation.png; leaving existing PNG untouched"); return
    os.makedirs(f"{OUT_ROOT}/{out_subdir}", exist_ok=True)
    x = np.arange(len(sims)); width = 0.25
    fig, ax = plt.subplots(figsize=(8, 6))
    for k, label, color, off in [
        ("start",  "% Collapsed at Start", "#1f77b4", -width),
        ("end",    "% Collapsed at End",   "#ff7f0e", 0.0),
        ("rounds", "% Rounds Collapsed",   "#2ca02c", width)]:
        p = np.array(pts[k]); lo = np.array(los[k]); hi = np.array(his[k])
        yerr = np.vstack([p - lo, hi - p])  # asymmetric 95% Wilson CI
        bars = ax.bar(x + off, p, width, yerr=yerr, capsize=4, label=label, color=color)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                    f"{bar.get_height():.1f}", ha="center", va="bottom",
                    fontsize=9, fontweight="bold", color=color)
    ax.set_ylabel("%", fontsize=12)
    ax.set_title("Collapse by Simulation (error bars: 95% Wilson CI)", fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{s.lower()}\nsimulation" for s in sims], fontsize=10)
    ax.legend(loc="upper right")
    ax.set_ylim(0, 115)
    plt.tight_layout()
    plt.savefig(f"{OUT_ROOT}/{out_subdir}/collapse_by_simulation.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"wrote {out_subdir}/collapse_by_simulation.png: {sims}")

MT = "model_collapse_log_graphite"
QM = "Qwen_Qwen2.5-14B-Instruct"
_BV = [("Replace All", "replace_all"), ("Replace One", "replace_one"), ("Search", "search")]

# 1) baseline overlays + collapse-by-simulation, 4 models
for subdir, mtok, extra in [
    ("Qwen/Qwen2.5-14B-Instruct", "Qwen_Qwen2.5-14B-Instruct", ""),
    ("meta-llama/Llama-3.1-8B-Instruct", "meta-llama_Llama-3.1-8B-Instruct", ""),
    ("mistralai/Mistral-7B-Instruct-v0.3", "mistralai_Mistral-7B-Instruct-v0.3", "paraphrase_on_"),
    ("deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "deepseek-ai_DeepSeek-R1-Distill-Qwen-7B", "")]:
    series = [(lab, COLORS[lab], f"{MT}_baseline_{rt}_{mtok}_{extra}local_{rt}.entities_by_round.jsonl")
              for lab, rt in _BV]
    plot_group(series, f"{subdir}/baseline")
    plot_collapse_group(series, f"{subdir}/baseline")

# 2) comparison: Qwen baseline (RA/RO/Search) + Agentic RAG
comparison_series = [
    ("Replace All", COLORS["Replace All"], f"{MT}_baseline_replace_all_{QM}_local_replace_all.entities_by_round.jsonl"),
    ("Replace One", COLORS["Replace One"], f"{MT}_baseline_replace_one_{QM}_local_replace_one.entities_by_round.jsonl"),
    ("Search", COLORS["Search"], f"{MT}_baseline_search_{QM}_local_search.entities_by_round.jsonl"),
    ("Agentic RAG", COLORS["Agentic RAG"], f"{MT}_agentic_rag_{QM}_local_agentic_rag.entities_by_round.jsonl")]
plot_group(comparison_series, "Qwen/Qwen2.5-14B-Instruct/comparison")
plot_collapse_group(comparison_series, "Qwen/Qwen2.5-14B-Instruct/comparison")

# 3) rerun-paraphrase: Qwen paraphrase RA/RO/Search
para_series = [
    ("Replace All", COLORS["Replace All"], f"{MT}_paraphrase_hybrid_{QM}_rerun-paraphrase_cpu_hybrid_paraphrased.entities_by_round.jsonl"),
    ("Replace One", COLORS["Replace One"], f"{MT}_paraphrase_replace_one_{QM}_rerun-paraphrase_cpu_replace_one_paraphrased.entities_by_round.jsonl"),
    ("Search", COLORS["Search"], f"{MT}_paraphrase_search_{QM}_rerun-paraphrase_cpu_search_paraphrased.entities_by_round.jsonl")]
plot_group(para_series, "Qwen/Qwen2.5-14B-Instruct/rerun-paraphrase")
plot_collapse_group(para_series, "Qwen/Qwen2.5-14B-Instruct/rerun-paraphrase")

# 4) rerank (Qwen): full λ-sweep (0.1 / 0.5 / 0.7 oracle) -> rerank/
rerank_sweep = [
    ("Rerank λ=0.1", LAMBDA_COLORS["Rerank λ=0.1"], f"{MT}_rerank_{QM}_rerank_lambda0.1.entities_by_round.jsonl"),
    ("Rerank λ=0.5", LAMBDA_COLORS["Rerank λ=0.5"], f"{MT}_rerank_{QM}_rerank_lambda0.5.entities_by_round.jsonl"),
    ("Rerank λ=0.7 (oracle)", LAMBDA_COLORS["Rerank λ=0.7 (oracle)"], f"{MT}_rerank_{QM}_rerank_lambda0.7_oracle.entities_by_round.jsonl")]
plot_group(rerank_sweep, "Qwen/Qwen2.5-14B-Instruct/rerank")
plot_collapse_group(rerank_sweep, "Qwen/Qwen2.5-14B-Instruct/rerank")

# 4b) rerank (Qwen) λ=0.7 focus: oracle vs desklib -> rerank_lambda0.7/  (matches the non-Qwen rerank panels)
rerank_q07 = [
    ("Rerank λ=0.7 (oracle)", LAMBDA_COLORS["Rerank λ=0.7 (oracle)"], f"{MT}_rerank_{QM}_rerank_lambda0.7_oracle.entities_by_round.jsonl"),
    ("Rerank λ=0.7 (desklib)", LAMBDA_COLORS["Rerank λ=0.7 (desklib)"], f"{MT}_rerank_{QM}_rerank_lambda0.7_desklib.entities_by_round.jsonl")]
plot_group(rerank_q07, "Qwen/Qwen2.5-14B-Instruct/rerank_lambda0.7")
plot_collapse_group(rerank_q07, "Qwen/Qwen2.5-14B-Instruct/rerank_lambda0.7")

# 5) agentic_rag standalone (Qwen) -> line charts only (single simulation; collapse-by-sim n/a)
plot_group(
    [("Agentic RAG", COLORS["Agentic RAG"], f"{MT}_agentic_rag_{QM}_local_agentic_rag.entities_by_round.jsonl")],
    "Qwen/Qwen2.5-14B-Instruct/agentic_rag")


Using entity re-run folder: entity re-run for workshop paper-20260629T230340Z-3-001\entity re-run for workshop paper


wrote Qwen/Qwen2.5-14B-Instruct/baseline: ['Replace All', 'Replace One', 'Search']


wrote Qwen/Qwen2.5-14B-Instruct/baseline/collapse_by_simulation.png: ['Replace All', 'Replace One', 'Search']


wrote meta-llama/Llama-3.1-8B-Instruct/baseline: ['Replace All', 'Replace One', 'Search']


wrote meta-llama/Llama-3.1-8B-Instruct/baseline/collapse_by_simulation.png: ['Replace All', 'Replace One', 'Search']


...

wrote Qwen/Qwen2.5-14B-Instruct/rerank: ['Rerank λ=0.1', 'Rerank λ=0.5', 'Rerank λ=0.7 (oracle)']


wrote Qwen/Qwen2.5-14B-Instruct/rerank/collapse_by_simulation.png: ['Rerank λ=0.1', 'Rerank λ=0.5', 'Rerank λ=0.7 (oracle)']


wrote Qwen/Qwen2.5-14B-Instruct/rerank_lambda0.7: ['Rerank λ=0.7 (oracle)', 'Rerank λ=0.7 (desklib)']


wrote Qwen/Qwen2.5-14B-Instruct/rerank_lambda0.7/collapse_by_simulation.png: ['Rerank λ=0.7 (oracle)', 'Rerank λ=0.7 (desklib)']


wrote Qwen/Qwen2.5-14B-Instruct/agentic_rag: ['Agentic RAG']


In [2]:
# ── Appendix panels (new dump has the data): non-Qwen rerank + non-Qwen paraphrase ──
# Non-Qwen models only have lambda=0.7 in this dump, in oracle + desklib detector variants,
# so the rerank panel compares oracle vs desklib at lambda=0.7 (not a lambda sweep like Qwen).

# rerank: DeepSeek / Llama / Mistral -> <org>/<model>/rerank/  (oracle vs desklib @ λ=0.7)
for subdir, mtok in [
    ("deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "deepseek-ai_DeepSeek-R1-Distill-Qwen-7B"),
    ("meta-llama/Llama-3.1-8B-Instruct", "meta-llama_Llama-3.1-8B-Instruct"),
    ("mistralai/Mistral-7B-Instruct-v0.3", "mistralai_Mistral-7B-Instruct-v0.3")]:
    series = [
        ("Rerank λ=0.7 (oracle)", LAMBDA_COLORS["Rerank λ=0.7 (oracle)"],
         f"{MT}_rerank_{mtok}_rerank_lambda0.7_oracle.entities_by_round.jsonl"),
        ("Rerank λ=0.7 (desklib)", LAMBDA_COLORS["Rerank λ=0.7 (desklib)"],
         f"{MT}_rerank_{mtok}_rerank_lambda0.7_desklib.entities_by_round.jsonl")]
    plot_group(series, f"{subdir}/rerank")
    plot_collapse_group(series, f"{subdir}/rerank")

# paraphrase: DeepSeek / Llama / Mistral -> <org>/<model>/rerun-paraphrase/  (RA/RO/Search)
for subdir, mtok in [
    ("deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "deepseek-ai_DeepSeek-R1-Distill-Qwen-7B"),
    ("meta-llama/Llama-3.1-8B-Instruct", "meta-llama_Llama-3.1-8B-Instruct"),
    ("mistralai/Mistral-7B-Instruct-v0.3", "mistralai_Mistral-7B-Instruct-v0.3")]:
    series = [
        ("Replace All", COLORS["Replace All"],
         f"{MT}_paraphrase_hybrid_{mtok}_rerun-paraphrase_cpu_hybrid_paraphrased.entities_by_round.jsonl"),
        ("Replace One", COLORS["Replace One"],
         f"{MT}_paraphrase_replace_one_{mtok}_rerun-paraphrase_cpu_replace_one_paraphrased.entities_by_round.jsonl"),
        ("Search", COLORS["Search"],
         f"{MT}_paraphrase_search_{mtok}_rerun-paraphrase_cpu_search_paraphrased.entities_by_round.jsonl")]
    plot_group(series, f"{subdir}/rerun-paraphrase")
    plot_collapse_group(series, f"{subdir}/rerun-paraphrase")


wrote deepseek-ai/DeepSeek-R1-Distill-Qwen-7B/rerank: ['Rerank λ=0.7 (oracle)', 'Rerank λ=0.7 (desklib)']


wrote deepseek-ai/DeepSeek-R1-Distill-Qwen-7B/rerank/collapse_by_simulation.png: ['Rerank λ=0.7 (oracle)', 'Rerank λ=0.7 (desklib)']


wrote meta-llama/Llama-3.1-8B-Instruct/rerank: ['Rerank λ=0.7 (oracle)', 'Rerank λ=0.7 (desklib)']


wrote meta-llama/Llama-3.1-8B-Instruct/rerank/collapse_by_simulation.png: ['Rerank λ=0.7 (oracle)', 'Rerank λ=0.7 (desklib)']


wrote mistralai/Mistral-7B-Instruct-v0.3/rerank: ['Rerank λ=0.7 (oracle)', 'Rerank λ=0.7 (desklib)']


...

wrote deepseek-ai/DeepSeek-R1-Distill-Qwen-7B/rerun-paraphrase/collapse_by_simulation.png: ['Replace All', 'Replace One', 'Search']


wrote meta-llama/Llama-3.1-8B-Instruct/rerun-paraphrase: ['Replace All', 'Replace One', 'Search']


wrote meta-llama/Llama-3.1-8B-Instruct/rerun-paraphrase/collapse_by_simulation.png: ['Replace All', 'Replace One', 'Search']


wrote mistralai/Mistral-7B-Instruct-v0.3/rerun-paraphrase: ['Replace All', 'Replace One', 'Search']


wrote mistralai/Mistral-7B-Instruct-v0.3/rerun-paraphrase/collapse_by_simulation.png: ['Replace All', 'Replace One', 'Search']


In [3]:
# ── Coverage check: every *.entities_by_round.jsonl in the dump must be plotted ──
all_files = {f for f in os.listdir(BASE) if f.endswith(".entities_by_round.jsonl")}
uncovered = sorted(all_files - USED)
print(f"dump files: {len(all_files)}   plotted: {len(USED)}   uncovered: {len(uncovered)}")
for f in uncovered:
    print("  UNCOVERED:", f)
assert not uncovered, f"{len(uncovered)} dump file(s) not plotted — see list above"
print("OK: every file in the dump is covered.")


dump files: 35   plotted: 35   uncovered: 0
OK: every file in the dump is covered.
